# World 行の抽出（2000〜2025年）

`01_prepare_data.ipynb` が取得した624ブロック（24類 × 26年）から、
**`partnerDesc == "World"` の行だけ**を抜き出す。

`World` は「その報告国がその品目を全世界へ輸出した合計」を表す集計行。
相手国別の内訳を持たないので、**各国の総輸出額を品目ごとに見たいとき**に使う。

| | |
|---|---|
| 入力 | `data/ex_ch01-24_2000-2025_allp/ch{類}_{年}.csv.gz` 624ブロック |
| 出力 | `data/world_rows/` に年ごと26ファイル + 全年結合1ファイル |
| 規模 | 全体の約7%（2022年の実測: 991,563行中 69,878行） |

> **注意 — 内訳と合算しないこと。** `World` 行は個別相手国の合計に等しいので、
> 両方を足すと必ず二重計上になる。この抽出データは `World` のみなので単独で使う分には安全。

## 1. セットアップ

In [ ]:
import time
from pathlib import Path

import pandas as pd

from settings import YEARS, CHAPTERS, BLOCK_DIR, WORLD_DIR, PERIOD_TAG

SRC = BLOCK_DIR
OUT = WORLD_DIR

# dtype 指定は必須。省くと 0201 が 201 になり先頭ゼロが落ちる。
DTYPE = {"cmdCode": str, "classificationCode": str}

if not SRC.exists():
    raise FileNotFoundError(f"入力フォルダが無い: {SRC}")

OUT.mkdir(parents=True, exist_ok=True)
print("入力:", SRC.resolve())
print("出力:", OUT.resolve())
print(f"対象: {YEARS[0]}〜{YEARS[-1]}年 × 第{CHAPTERS[0]}〜{CHAPTERS[-1]}類 = {len(YEARS)*len(CHAPTERS)} ブロック")

## 2. 抽出

年ごとに24類を読み、`World` 行だけを残して保存する。
**保存済みの年はスキップ**するので、途中で止めても再実行すれば続きから走る。

各ブロック内は `cmdCode` 昇順だが、24類を連結すると類の境界で並びが切れるため、
結合後に `cmdCode` → 報告国 で並べ直す。

In [ ]:
t_all = time.time()
summary = []

for year in YEARS:
    out_path = OUT / f"world_{year}.csv.gz"
    if out_path.exists():
        print(f"[skip] {year} — 保存済み")
        continue

    t0 = time.time()
    paths = [SRC / f"ch{ch}_{year}.csv.gz" for ch in CHAPTERS]
    lack = [p.name for p in paths if not p.exists()]
    if lack:
        raise FileNotFoundError(f"{year}: ブロックが欠けている {lack}")

    frames = []
    for p in paths:
        with pd.read_csv(p, dtype=DTYPE, chunksize=100_000) as reader:
            for d in reader:
                frames.append(d.loc[d["partnerDesc"] == "World"].copy())
    df_y = pd.concat(frames, ignore_index=True)

    if df_y.empty:
        raise RuntimeError(f"{year}: World 行が1件も無い。抽出条件を確認すること。")

    df_y = (df_y.sort_values(["cmdCode", "reporterCode"], kind="stable")
                .reset_index(drop=True))
    df_y.to_csv(out_path, index=False, encoding="utf-8-sig", compression="gzip")

    el = time.time() - t0
    print(f"{year}: {len(df_y):>8,} 行 / {out_path.stat().st_size/1e6:5.1f} MB / {el:5.1f}秒")
    summary.append((year, len(df_y), round(el, 1)))

print(f"\n完了: {time.time()-t_all:.0f} 秒")
print(f"出力: {len(list(OUT.glob('world_*.csv.gz')))} ファイル")
if summary:
    s = pd.DataFrame(summary, columns=["year", "rows", "sec"])
    print(f"今回抽出: {s['rows'].sum():,} 行")
    display(s)

## 3. 全年を1ファイルに結合

In [ ]:
# 年ごとのファイルのみを結合する（前回の全年結合ファイルは含めない）。
files = [OUT / f"world_{year}.csv.gz" for year in YEARS]
missing = [p.name for p in files if not p.exists()]
if missing:
    raise FileNotFoundError(f"年別ファイルが不足しています: {missing}。セクション2を実行してください。")
print(f"{len(files)} ファイルを結合")

world = pd.concat([pd.read_csv(f, dtype=DTYPE) for f in files], ignore_index=True)
world = (world.sort_values(["period", "cmdCode", "reporterCode"], kind="stable")
              .reset_index(drop=True))

ALL = OUT / f"world_{PERIOD_TAG}_all.csv.gz"
world.to_csv(ALL, index=False, encoding="utf-8-sig", compression="gzip")
print(f"保存: {ALL.resolve()}")
print(f"      {len(world):,} 行 x {world.shape[1]} 列 / {ALL.stat().st_size/1e6:.1f} MB")

## 4. 検証

In [ ]:
print("相手国       :", sorted(world["partnerDesc"].astype(str).unique()), " ← World のみ")
print("partnerCode  :", sorted(world["partnerCode"].unique()), " ← 0 のみ")
print("フロー       :", sorted(world["flowDesc"].astype(str).unique()))
print("年           :", f"{world['refYear'].min()}〜{world['refYear'].max()}（{world['refYear'].nunique()}年）")
print("品目コード桁数:", sorted(world["cmdCode"].astype(str).str.len().unique()), " ← [6]")
print("類の範囲     :", f"{world['cmdCode'].str[:2].min()}〜{world['cmdCode'].str[:2].max()}")
print("報告国数     :", world["reporterCode"].nunique())
print("品目数       :", world["cmdCode"].nunique())

dup = world.duplicated(subset=["period", "reporterCode", "cmdCode", "flowDesc"]).sum()
print("重複行       :", dup, " ← 0 であること")
if dup:
    raise RuntimeError("重複がある。抽出元に内訳行が混入している可能性。")

In [ ]:
# 年ごとの行数と報告国数
world.groupby("refYear").agg(行数=("cmdCode", "size"),
                            報告国=("reporterCode", "nunique"),
                            品目=("cmdCode", "nunique"))

In [ ]:
# 中身のサンプル
VIEW = ["period", "reporterCode", "reporterDesc", "flowDesc",
        "partnerCode", "partnerDesc", "cmdCode", "cmdDesc",
        "qty", "qtyUnitAbbr", "netWgt", "primaryValue"]
world[[c for c in VIEW if c in world.columns]].head(20)

## メモ

- **`World` 行だけなので、相手国別の分析はできない。** 相手国内訳が必要なら
  `data/ex_ch01-24_2000-2025_allp/` の元ブロックを使う。
- **輸出額は `primaryValue`（= FOB）を使う。** `cifvalue` は輸出データではほぼ空。
- **直近年は報告が出揃っていない。** 報告国数が2024年以降で大きく落ちるため、
  時系列比較でそのまま使うと過小評価になる。
- 読み込みは `dtype` 指定を忘れないこと。

  ```python
  pd.read_csv(path, dtype={"cmdCode": str, "classificationCode": str})
  ```